In [98]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 16
batch_size = 32
max_iters = 5000
learning_rate = 2e-5
eval_iters = 250
n_embed = 192 # 384
n_head = 8
n_layer = 8
dropout = 0.2

In [99]:
chars = ""

with open('data/dutch_bible.txt', 'r', encoding='utf8') as file:
    text = file.read()
    
chars = sorted(set(text))
vocabulary_size = len(chars)
print(vocabulary_size)

88


In [100]:
string_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_string = { i:ch for i, ch in enumerate(chars) }

encode = lambda s: [string_to_int[ch] for ch in s]
decode = lambda x: ''.join([int_to_string[i] for i in x])

data = torch.tensor(encode(text), dtype=torch.long)

In [101]:
# Defines the split (80% train, 20% validation)
n = int(0.8 * len(data))
train_data = data[:n]
validation_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else validation_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')

In [102]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)  
        q = self.query(x)
        wgt = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wgt = wgt.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wgt = F.softmax(wgt, dim=-1)
        wgt = self.dropout(wgt)
        v = self.value(x)
        out = wgt @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
    

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),                # Dropout drops the percentage of the result to avoid overfitting
        )

    def forward(self, x):
        return self.net(x)
    
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embed, n_head):
        # n_embed: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, n_embed)
        self.positional_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*(Block(n_embed, n_head=n_head) for _ in range(n_layer)))
        
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocabulary_size)
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        
    def forward(self, index, targets=None):
        B, T = index.shape
        
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(index) # (B,T,C)
        pos_emb = self.positional_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, loss = self.forward(index_cond)
            logits = logits[:, -1, :] # becomes (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index
    
model = GPTLanguageModel(vocabulary_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(context)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)

tensor([[0]])



hw9ü+=Tï&:B(j
JW>QWïUS+ l/ "Vgwyj;6gW
0vDGJéaTzH8è"JBJ3tòDAc6c7mZïY n`UkéL>4fü4vU IWVRTy 5ä-4v0R.79!


In [103]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [104]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'Iter {iter}, train loss: {losses["train"]}, val loss: {losses["val"]}')
    
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    # Optimises and sets the gradients to zero so more efficient because int instead of float
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

torch.save(model.state_dict(), 'model-gpt-v1.pth')

Iter 0, train loss: 4.502331256866455, val loss: 4.5017781257629395
Iter 250, train loss: 3.1084280014038086, val loss: 3.0904829502105713
Iter 500, train loss: 2.741866111755371, val loss: 2.7224178314208984
Iter 750, train loss: 2.57047176361084, val loss: 2.554206132888794
Iter 1000, train loss: 2.479275703430176, val loss: 2.458986520767212
Iter 1250, train loss: 2.4079365730285645, val loss: 2.3979601860046387
Iter 1500, train loss: 2.3438971042633057, val loss: 2.339186906814575
Iter 1750, train loss: 2.2989087104797363, val loss: 2.287684917449951
Iter 2000, train loss: 2.2671947479248047, val loss: 2.252237319946289
Iter 2250, train loss: 2.231672525405884, val loss: 2.2263023853302
Iter 2500, train loss: 2.206104278564453, val loss: 2.200227737426758
Iter 2750, train loss: 2.179788112640381, val loss: 2.1654672622680664
Iter 3000, train loss: 2.1507885456085205, val loss: 2.1497642993927
Iter 3250, train loss: 2.13173246383667, val loss: 2.12123966217041
Iter 3500, train loss:

In [107]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=100)[0].tolist())
print(generated_chars)


: Ik: Diegegea hend: "`pecht iek do+ken, leten ein zelt.
17: VIksneringen aan Her die erin te barlev
